# Project Methodology

This notebook documents the methodology of the traffic signal control project before presenting results. The focus is to make the full experimental setup explicit: environment, state and action definitions, modular architectures, PPO training setup, Optuna search protocol, and evaluation methodology.

## Scope

- Primary maintained search path: **PPO with modular CNN+recurrent architectures**
- Comparison baseline in the repository: **Rainbow DQN**
- Environment: **single SUMO intersection** with RL-based traffic signal control
- Study objective: select architectures and hyperparameters that **generalize across traffic demand levels**, not only the training demand


## 1. Problem Setting

The project studies **traffic signal control through deep reinforcement learning** in a SUMO environment. The control agent selects signal phases for a single intersection while observing a spatial representation of incoming traffic. The central research question is not only whether an agent can optimize one fixed traffic pattern, but whether the learned control policy can remain effective under **different traffic intensities**. This is why the final Optuna objective is based on **multi-demand evaluation**.

## 2. Environment

### Simulator

- **SUMO** is used as the traffic simulator
- The scripts interact with SUMO through **TraCI-compatible interfaces**
- In headless execution, the code prefers **libsumo** when available because it is faster than socket-based TraCI

### Intersection Setup

The maintained configuration uses the SUMO assets under `code/intersection/`, especially:

- `environment.net.xml`
- `sumo_config.sumocfg`
- `tls.add.xml`
- `episode_routes.rou.xml` (generated per episode)

### Episode Horizon

For the maintained PPO search path:

- Maximum simulation steps per episode: **3600**
- Green duration used during training/evaluation: **7**
- Yellow duration used during training/evaluation: **6**
- Total training episodes per trial: **800**

## 3. Traffic Demand Generation

Traffic demand is generated episode by episode using `generator.py`.

### Training demand generation

- Number of generated vehicles per training episode: **1000**
- Maintained traffic distribution in the current PPO study: **Weibull**
- A fixed seed schedule is used across episodes so that different Optuna trials see the **same sequence of traffic realizations**

### Why this matters

Using a fixed seed schedule reduces one source of variance when comparing architectures. This makes the search closer to a controlled study of model design and training hyperparameters.

## 4. State Representation

The agent does not observe raw vehicle lists directly. Instead, the intersection is encoded as a **3-channel spatial tensor**.

### Full spatial map

The environment first builds a fixed occupancy canvas of shape **209 x 206** so that each incoming lane is always mapped to a fixed spatial location.

### Cropped observation used by the neural network

The final observation passed to the policy/value networks is a centered crop of shape:

- **3 x 48 x 46**

### Channels

- Channel 0: **vehicle presence / occupancy**
- Channel 1: **normalized speed**
- Channel 2: **normalized accumulated waiting time**

This design turns the traffic state into an image-like input while preserving lane geometry around the intersection.

## 5. Action Space

The maintained action space is discrete with **8 actions**, each representing one admissible green phase.

### Action definitions

- Action 0: North-South green
- Action 1: North-South-left green
- Action 2: East-West green
- Action 3: East-West-left green
- Action 4: North straight + left special phase
- Action 5: East straight + left special phase
- Action 6: South straight + left special phase
- Action 7: West straight + left special phase

### Transition rule

If the selected action differs from the previous one, the controller first applies the corresponding **yellow phase** and only then applies the new green phase.

This makes action execution closer to a realistic traffic controller and prevents unrealistically abrupt signal switching.

## 6. Reward Definition

The maintained PPO training path uses a queue-based shaped reward.

For each control decision:

- Let `last_queue` be the queue length before applying the selected phase
- Let `new_queue` be the queue length after simulating the phase duration

The reward is:

```text
reward = (last_queue - new_queue) - 0.2 * new_queue
```

### Interpretation

- Positive reward when queue length is reduced
- Negative reward when queue length remains high or increases
- The extra `-0.2 * new_queue` term penalizes persistent congestion

This reward is local and dense, which is useful for PPO optimization in a long-horizon traffic control task.

## 7. Modular PPO Architecture

The maintained PPO models are defined in `networks.py` and use a modular **CNN + recurrent layer + MLP** structure.

### Shared design idea

Both actor and critic receive the same `3 x 48 x 46` observation format and extract:

1. **Spatial features** with a configurable convolutional stack
2. **Temporal context** with a configurable recurrent layer (LSTM or GRU)
3. **Decision/value features** with MLP layers

### Actor

- Convolutional feature extractor
- Recurrent memory module (LSTM or GRU)
- MLP head
- Final softmax over the **8 discrete traffic phases**

### Critic

- Same style of convolutional backbone
- Recurrent memory module (LSTM or GRU)
- MLP head
- Final scalar value estimate

### Why this modularization matters

This structure makes it possible to study how much spatial depth, temporal memory, and MLP capacity are needed for robust control across different traffic conditions.

## 8. PPO Hyperparameters

### Fixed training setup in the maintained study

- Observation shape: **[3, 48, 46]**
- Number of actions: **8**
- Episode length: **3600** steps
- Rollout horizon `T_horizon`: **256**
- Batch size: **16**
- Advantage normalization: **enabled**
- Training episodes per trial: **800**

### Optimized PPO hyperparameters

The Optuna search samples:

- Learning rate
- Number of PPO epochs `K_epochs`
- L2 regularization
- Discount factor `gamma`
- GAE parameter `lambda`
- PPO clip range
- Entropy coefficient
- Optimizer choice (`adam` or `adamw`)
- Weight decay when `adamw` is selected

## 9. Architecture Search Space

The Optuna trial also samples architectural hyperparameters.

### Convolutional block

- Number of convolutional layers: **1 to 2**
- Filters per layer: **16, 32, 64, 128, 256**
- Pooling stride per layer: **1 to 3**
- Kernel sizes: **3 to 9** with step 2

### Recurrent block

- LSTM units: **16, 32, 64, 96, 128, 256**

### MLP block

- Number of MLP layers: **2 to 3**
- Hidden units per MLP layer: **32, 64, 128**

This means each trial defines both a **model architecture** and a **training configuration**.

## 10. What One Optuna Trial Consists Of

A single Optuna trial includes:

1. Sampling one architecture and one PPO hyperparameter configuration
2. Building the actor and critic with that configuration
3. Training the agent for **800 episodes**
4. Optionally pruning weak runs using an intermediate evaluation signal
5. Evaluating the fully trained policy across **multiple traffic demand levels**
6. Returning the final weighted score to Optuna

This is important because the study is searching for **generalizable RL configurations**, not simply fitting a static benchmark.

## 11. Intermediate Pruning Strategy

During training, the script periodically performs a pruning evaluation.

### Current policy

- Start pruning checks after episode **400**
- Re-check every **20** episodes
- Use a **weighted intermediate evaluation** over demands **1000**, **1300**, and **1600** vehicles/hour
- Use a pruning seed block that is separate from both training and final evaluation
- Report the aggregate score to Optuna with `trial.report(...)`
- Allow Optuna's `MedianPruner` to stop weak configurations early

## 12. Final Generalization Evaluation

The most important design choice for the study is the **final multi-demand evaluation**.

### Evaluation demand levels

The trained PPO policy is tested on traffic volumes:

- **1000, 1100, 1200, ..., 2000** vehicles/hour

### Repetitions per demand

- **10 evaluation turns** per demand level

### Final Optuna objective

The final score returned to Optuna is the **weighted average** of the scores across the demand sweep. This means a configuration is preferred if it performs well **across a range of traffic loads**, not merely at the nominal training demand.

## 13. Evaluation Metrics

During final evaluation, the project records more than the final scalar objective.

### Primary scalar used by Optuna

- **Weighted average score** across the evaluated traffic volumes

### Per-demand evaluation outputs

For each demand level, the code logs:

- `score`
- `avg_speed`
- `cumulative_wait`
- `avg_queue_length`

### Recently added logging

The PPO script now also stores:

- `eval/demand_levels`
- `eval/score_vector`

This preserves the full performance curve across demand levels, not only the weighted mean.

## 14. What the Score Means vs. Traffic Metrics

The code distinguishes between the RL score used for model selection and interpretable operational traffic metrics such as average speed, cumulative waiting time, and average queue length. This separation is useful in a paper because the RL objective and the transportation metrics do not always align perfectly.

## 15. Optuna Configuration

The maintained study runner uses:

- **TPESampler** with a fixed seed
- **MedianPruner**
- persistent **SQLite storage**
- `load_if_exists=True` so a study can be resumed

This setup is practical for long RL studies because it is resumable and can prune clearly underperforming trials.

## 16. Experiment Logging and Reproducibility

### Logging

The maintained scripts use **Weights & Biases (wandb)** to log:

- trial configuration
- training curves
- pruning evaluations
- final multi-demand evaluation metrics

### Reproducibility practices already present

- fixed episode seed schedule across trials
- script-based experiment path for the maintained PPO study
- persistent Optuna storage
- explicit recording of architecture and hyperparameters in W&B

### Remaining practical caveat

RL experiments still contain variance from optimization, simulator interactions, and hardware/software differences. The project is reproducible in structure and protocol, but not deterministic in the strict sense.

## 17. Repository Components Relevant to the Methodology

### Core maintained files

- `SignalTrafficOptimization.py`: PPO training + Optuna search
- `simulation.py`: SUMO interaction loop and reward execution logic
- `networks.py`: modular actor/critic architecture
- `generator.py`: traffic-demand generation
- `utils.py`: SUMO setup and project paths

### Comparison baseline

- `SignalTrafficOptimization_Rainbow.py`
- `rainbow_networks.py`

### Notebooks

The notebooks remain useful for exploration and explanation, but the maintained search path for PPO is the script-based pipeline.

## 18. Methodological Strengths of the Study

From a research perspective, the setup has several strengths:

- It evaluates **generalization across traffic demand levels** instead of only one fixed training condition
- It searches over **architecture and optimization hyperparameters jointly**
- It separates **training reward** from **traffic performance metrics**
- It uses a spatial-temporal architecture that is well matched to traffic state structure
- It supports resumable long-running studies through Optuna + SQLite

These points make the project stronger than a simple single-scenario RL benchmark.

## 19. Items to Add Later in the Results Section

This notebook is intentionally methodology-first. The next step is to add results, for example:

- best PPO trial configuration
- parameter importance analysis from Optuna
- score vs. demand curves for top trials
- queue/wait/speed trade-offs
- in-distribution vs. out-of-distribution demand behavior
- comparison against Rainbow DQN and/or fixed-time baselines
- qualitative discussion of which architecture choices improve generalization

## 20. Possible Missing Methodological Items to Consider

If the methodology is later turned into a paper section, the following may also be useful to state explicitly:

- hardware used for search and approximate runtime budget
- whether demand distributions beyond Weibull are used in any reported experiment
- whether multiple random seeds are used for the final top-model comparison
- whether evaluation traffic volumes are strictly out of distribution relative to training
- whether a classical baseline (fixed-time, actuated, Webster, etc.) is included

These are not all required now, but they are worth deciding before writing the final results section.
